In [1]:
import pandas as pd

# Set pandas display options
pd.set_option('display.max_columns', None)  # Show all columns
pd.set_option('display.width', 100)        # Set the display width to 1000 characters
pd.set_option('display.max_colwidth', None) # Allow the full content of each column to be displayed

In [2]:
df_final_audit = pd.read_csv('./comprehensive_audit_final.csv')

In [3]:
import os, json, time
import pandas as pd
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── 설정 ─────────────────────────────────────────────────
load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
MODEL = "gpt-4o-mini"

df_final_audit = pd.read_csv('./sfscore_df.csv', encoding='utf-8-sig')
all_vars = [col.replace('원본_', '') for col in df_final_audit.columns if col.startswith('원본_')]

np.random.seed(42)
df_sample = df_final_audit.sample(n=100).reset_index(drop=True)
print(f"샘플: {len(df_sample)}건 | 동일 입력 2회 반복 실행")

# ── 프롬프트 (현행 A 기준) ───────────────────────────────
SYSTEM = (
    "당신은 보험 언더라이팅 전문가입니다. "
    "주어진 의료 소견서에서 아래 변수들을 정확히 추출하여 JSON 형식으로만 반환하십시오. "
    "다른 텍스트는 출력하지 마십시오."
)
def make_prompt(sogyeon):
    return (
        f"다음 의료 소견서에서 아래 변수들을 추출하여 JSON으로 반환하십시오.\n"
        f"변수 목록: {all_vars}\n\n소견서:\n{sogyeon}\n\n"
        f'응답 형식: {{"변수명": "값", ...}}'
    )

# ── 추출 함수 ────────────────────────────────────────────
def extract_one(idx_row):
    idx, row = idx_row
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": SYSTEM},
                {"role": "user", "content": make_prompt(row['합성_소견서'])}
            ],
            temperature=0,
            max_tokens=1024,
        )
        raw = response.choices[0].message.content.strip()
        raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
        return idx, json.loads(raw), None
    except Exception as e:
        return idx, {}, str(e)

# ── Run 1 실행 ───────────────────────────────────────────
print("\n▶ Run 1 실행 중...")
run1 = [None] * len(df_sample)
errors1 = 0
with ThreadPoolExecutor(max_workers=10) as ex:
    futures = {ex.submit(extract_one, (i, row)): i for i, row in df_sample.iterrows()}
    done = 0
    for f in as_completed(futures):
        idx, result, err = f.result()
        run1[idx] = result
        done += 1
        if err: errors1 += 1
        if done % 20 == 0:
            print(f"  {done}/100건 (오류: {errors1}건)", flush=True)
print(f"Run 1 완료. 오류: {errors1}건")

time.sleep(2)  # API 안정화

# ── Run 2 실행 ───────────────────────────────────────────
print("\n▶ Run 2 실행 중...")
run2 = [None] * len(df_sample)
errors2 = 0
with ThreadPoolExecutor(max_workers=10) as ex:
    futures = {ex.submit(extract_one, (i, row)): i for i, row in df_sample.iterrows()}
    done = 0
    for f in as_completed(futures):
        idx, result, err = f.result()
        run2[idx] = result
        done += 1
        if err: errors2 += 1
        if done % 20 == 0:
            print(f"  {done}/100건 (오류: {errors2}건)", flush=True)
print(f"Run 2 완료. 오류: {errors2}건")

# ── 재현성 비교 ──────────────────────────────────────────
exact_match = 0        # 전체 변수 완전 일치
partial_match = 0      # 일부 변수 불일치
var_match_rates = {v: [] for v in all_vars}

for i in range(len(df_sample)):
    r1, r2 = run1[i], run2[i]
    if not r1 or not r2:
        continue
    all_match = True
    for var in all_vars:
        v1 = str(r1.get(var, '')).strip()
        v2 = str(r2.get(var, '')).strip()
        match = (v1 == v2)
        var_match_rates[var].append(1 if match else 0)
        if not match:
            all_match = False
    if all_match:
        exact_match += 1
    else:
        partial_match += 1

total_valid = exact_match + partial_match

# ── 결과 출력 ────────────────────────────────────────────
print("\n" + "=" * 60)
print("재현성 검증 결과 (temperature=0, 동일 입력 2회 반복)")
print("=" * 60)
print(f"유효 샘플 수         : {total_valid}건")
print(f"완전 일치 (전 변수)  : {exact_match}건 ({exact_match/total_valid*100:.1f}%)")
print(f"부분 불일치          : {partial_match}건 ({partial_match/total_valid*100:.1f}%)")

print("\n▶ 변수별 일치율 (하위 5개)")
var_df = pd.DataFrame({
    '변수': list(var_match_rates.keys()),
    '일치율(%)': [np.mean(v)*100 if v else 0 for v in var_match_rates.values()]
}).sort_values('일치율(%)')
print(var_df.head().to_string(index=False))

print("\n▶ 변수별 평균 일치율")
print(f"  전체 변수 평균: {var_df['일치율(%)'].mean():.1f}%")
print(f"  연속형 변수 평균: {var_df[var_df['변수'].isin(all_vars[:10])]['일치율(%)'].mean():.1f}%")
print(f"  범주형 변수 평균: {var_df[var_df['변수'].isin(all_vars[10:])]['일치율(%)'].mean():.1f}%")

var_df.to_csv('reproducibility_result.csv', index=False, encoding='utf-8-sig')
print("\n결과 저장: reproducibility_result.csv")

샘플: 100건 | 동일 입력 2회 반복 실행

▶ Run 1 실행 중...
  20/100건 (오류: 0건)
  40/100건 (오류: 0건)
  60/100건 (오류: 0건)
  80/100건 (오류: 0건)
  100/100건 (오류: 0건)
Run 1 완료. 오류: 0건

▶ Run 2 실행 중...
  20/100건 (오류: 0건)
  40/100건 (오류: 0건)
  60/100건 (오류: 0건)
  80/100건 (오류: 0건)
  100/100건 (오류: 0건)
Run 2 완료. 오류: 0건

재현성 검증 결과 (temperature=0, 동일 입력 2회 반복)
유효 샘플 수         : 100건
완전 일치 (전 변수)  : 93건 (93.0%)
부분 불일치          : 7건 (7.0%)

▶ 변수별 일치율 (하위 5개)
       변수  일치율(%)
    체질량지수    95.0
     공복혈당    99.0
   총콜레스테롤    99.0
HDL-콜레스테롤    99.0
     중성지방    99.0

▶ 변수별 평균 일치율
  전체 변수 평균: 99.5%
  연속형 변수 평균: 99.0%
  범주형 변수 평균: 99.9%

결과 저장: reproducibility_result.csv
